In [1]:
import sys
import math
import numpy as np

sys.path.append('../build')
import demgpu


In [2]:
# Fixed Params
num_particles = 300
radius = 0.5
vol_particle = (4.0/3.0) * math.pi * radius**3
growth_rate = 1.0 # Slow growth

# Study Grid
# stiffness (iterations) vs timestep (dt)
# Higher iterations = stiffer solver, better overlap resolution
# Smaller dt = better stability, but slower growth

iterations_list = [100]
dt_list = [0.1, 0.01, 0.001, 0.0001]
T_gran = [0.0]

best_phi = 0.0
best_config = None

phi_ref = 1.0
vol_total_ref = num_particles * vol_particle
vol_domain_ref = vol_total_ref / phi_ref
domain_side = vol_domain_ref ** (1.0/3.0)

sim = demgpu.Simulation(num_particles)
sim.initialize(shape_type=1, radius=radius) # Sphere

half_d = domain_side / 2.0
sim.set_domain((-half_d, -half_d, -half_d), (half_d, half_d, half_d))
sim.set_gravity(0, 0, 0)
rng = np.random.default_rng(42)

print(f"Sphere Packing Optimization Study (N={num_particles})")
print(f"{'Iter':<5} {'dt':<8} {'TGran':<8} {'T':<8} {'MaxPhi':<8} {'FinalOv':<10} {'Notes'}")
print("-" * 50)

for iters in iterations_list:
    for dt in dt_list:
        for T in T_gran:    
            # Run one experiment
            
            # Target a high Phi to define domain

            sim.set_solver_iterations(iters, 0) # Only Position Solver Used for Overlap
        
            # Init Random
            pos = rng.uniform(-half_d, half_d, (num_particles, 4)).astype(np.float32)
            pos[:, 3] = 1.0
            sim.set_positions(pos)
            sigma = np.sqrt(T)
            vel = rng.normal(0, sigma, (num_particles, 4)).astype(np.float32)
            vel[:, 3] = 0.0
            sim.set_velocities(vel) 
            
            # Growth
            max_phi_reached = 0.0
            overlap_tolerance = 1e-4
            
            # Step limit? Or Growth limit?
            steps = 0
            limit_steps = np.ceil(1.0 / (dt * growth_rate))
            
            scales = np.zeros(num_particles, dtype=np.float32)
            
            final_ov = 0.0
            reason = "Limit"
            
            while steps < limit_steps:
                scale = min(0.05 + growth_rate*steps*dt, 1.0)
                scales[:] = scale
                sim.set_scales(scales)
                
                sim.step(dt)
                steps += 1
                
                current_ov = sim.get_max_overlap()
                final_ov = current_ov
                
                # Check Overlap
                if current_ov > overlap_tolerance:
                    # Try simple settle
                    settled = False
                    for _ in range(50): # Quick settle check
                            sim.step(dt)
                            if sim.get_max_overlap() < overlap_tolerance:
                                settled = True
                                break
                    
                    if not settled:
                        final_ov = sim.get_max_overlap()
                        reason = "Jam"
                        break
                
                # Update Max Phi (valid state)
                vel = sim.get_velocities()
                T_current = sum(vel[:, 0:3].ravel()**2) / (3*num_particles)
                phi_current = (num_particles * vol_particle * scale**3) / (domain_side**3)
                max_phi_reached = phi_current
                
                
            print(f"{iters:<5} {dt:<8.4f} {T:<8.4f} {T_current:<8.4f} {max_phi_reached:<8.3f} {final_ov:<10.3f} {reason}")
            
            if max_phi_reached > best_phi:
                best_phi = max_phi_reached
                best_config = (iters, dt)

print("-" * 50)
print(f"Best Result: Phi={best_phi:.3f} with Iter={best_config[0]}, dt={best_config[1]}")


Created Shape ID 0 via Manager. Points: 0
Simulation initialized. Seeding particles...
Domain Initialized: Periodicity enabled by default.
Sphere Packing Optimization Study (N=300)
Iter  dt       TGran    T        MaxPhi   FinalOv    Notes
--------------------------------------------------
100   0.1000   0.0000   0.0000   0.422    0.003      Jam
100   0.0100   0.0000   0.0000   0.551    0.000      Jam
100   0.0010   0.0000   0.0000   0.582    0.000      Jam
100   0.0001   0.0000   0.0000   0.598    0.000      Jam
--------------------------------------------------
Best Result: Phi=0.598 with Iter=100, dt=0.0001


In [19]:
# Packing of Hollow Cylinders
num_particles = 300

radius = 0.5
diameter = 1.0
height = 1.0 # H/D = 1
thickness = 0.3 # Arbitrary hollow thickness
# Calculate Domain Volume
# V_cyl = H * pi * (R^2 - r_in^2) = H * pi * (2Rt - t^2)
vol_particle = height * math.pi * (2 * radius * thickness - thickness**2)
growth_rate = 1.0 # Slow growth

# Study Grid
# stiffness (iterations) vs timestep (dt)
# Higher iterations = stiffer solver, better overlap resolution
# Smaller dt = better stability, but slower growth

iterations_list = [100]
dt_list = [0.1]
T_gran = [0.0]

best_phi = 0.0
best_config = None

phi_ref = 1.0
vol_total_ref = num_particles * vol_particle
vol_domain_ref = vol_total_ref / phi_ref
domain_side = vol_domain_ref ** (1.0/3.0)

sim = demgpu.Simulation(num_particles)
sim.initialize(shape_type=2, radius=radius, height=height, thickness=thickness) #hollow cylinder

half_d = domain_side / 2.0
sim.set_domain((-half_d, -half_d, -half_d), (half_d, half_d, half_d))
sim.set_gravity(0, 0, 0)
rng = np.random.default_rng(42)

print(f"Sphere Packing Optimization Study (N={num_particles})")
print(f"{'Iter':<5} {'dt':<8} {'TGran':<8} {'T':<8} {'MaxPhi':<8} {'FinalOv':<10} {'Notes'}")
print("-" * 50)

for iters in iterations_list:
    for dt in dt_list:
        for T in T_gran:    
            # Run one experiment
            
            # Target a high Phi to define domain

            sim.set_solver_iterations(iters, 10) # Only Position Solver Used for Overlap
        
            # Init Random
            pos = rng.uniform(-half_d, half_d, (num_particles, 4)).astype(np.float32)
            pos[:, 3] = 1.0
            sim.set_positions(pos)
            sigma = np.sqrt(T)
            vel = rng.normal(0, sigma, (num_particles, 4)).astype(np.float32)
            vel[:, 3] = 0.0
            sim.set_velocities(vel) 

            quat = np.zeros((num_particles, 4), dtype=np.float32)
            for i in range(num_particles):
                # Random axis
                u = rng.normal(0, 1, 3)
                u /= np.linalg.norm(u)
                angle = rng.uniform(0, math.pi*2)
                q = np.array([math.sin(angle/2)*u[0], math.sin(angle/2)*u[1], math.sin(angle/2)*u[2], math.cos(angle/2)])
                quat[i] = q
            sim.set_quaternions(quat)

            # Growth
            max_phi_reached = 0.0
            overlap_tolerance = 1e-7
            
            # Step limit? Or Growth limit?
            steps = 0
            limit_steps = np.ceil(1.0 / (dt * growth_rate))
            
            scales = np.zeros(num_particles, dtype=np.float32)
            
            final_ov = 0.0
            reason = "Limit"
            
            while steps < limit_steps:
                scale = min(0.05 + growth_rate*steps*dt, 1.0)
                scales[:] = scale
                sim.set_scales(scales)
                
                sim.step(dt)
                steps += 1
                
                current_ov = sim.get_max_overlap()
                final_ov = current_ov
                
                # Check Overlap
                if current_ov > overlap_tolerance:
                    # Try simple settle
                    settled = False
                    for _ in range(1000): # Quick settle check
                            sim.step(dt)
                            if sim.get_max_overlap() < overlap_tolerance:
                                settled = True
                                break
                    
                    if not settled:
                        final_ov = sim.get_max_overlap()
                        reason = "Jam"
                        break
                
                # Update Max Phi (valid state)
                vel = sim.get_velocities()
                T_current = sum(vel[:, 0:3].ravel()**2) / (3*num_particles)
                phi_current = (num_particles * vol_particle * scale**3) / (domain_side**3)
                max_phi_reached = phi_current
                
                
            print(f"{iters:<5} {dt:<8.4f} {T:<8.4f} {T_current:<8.4f} {max_phi_reached:<8.3f} {final_ov:<10.3f} {reason}")
            
            if max_phi_reached > best_phi:
                best_phi = max_phi_reached
                best_config = (iters, dt)

print("-" * 50)
print(f"Best Result: Phi={best_phi:.3f} with Iter={best_config[0]}, dt={best_config[1]}")


Created Shape ID 0 via Manager. Points: 675
Sphere Packing Optimization Study (N=300)
Iter  dt       TGran    T        MaxPhi   FinalOv    Notes
--------------------------------------------------
Simulation initialized. Seeding particles...
Domain Initialized: Periodicity enabled by default.
100   0.1000   0.0000   0.0000   0.275    0.000      Jam
--------------------------------------------------
Best Result: Phi=0.275 with Iter=100, dt=0.1


In [20]:
import os
os.makedirs("../output/sdf", exist_ok=True)
filename = f"../output/sdf/hollow_cylinders_packed_phi={max_phi_reached:.3f}.vti"
print(f"Exporting to {filename}...")
sim.export_sdf(filename, (256, 256, 256))
print("Success.")

Exporting to ../output/sdf/hollow_cylinders_packed_phi=0.275.vti...
SDF exported to ../output/sdf/hollow_cylinders_packed_phi=0.275.vti
Success.


In [ ]:
# Fixed Params
num_particles = 5
radius = 0.5
vol_particle = (4.0/3.0) * math.pi * radius**3

sim = demgpu.Simulation(num_particles)
sim.initialize(shape_type=1, radius=radius) # Sphere

L = num_particles * radius * 2.01
sim.set_domain((0.0, 0.0, 0.0), (L, L, L))
sim.set_gravity(0, 0, 0)

sim.set_solver_iterations(1, 0) # Only Position Solver Used for Overlap
pos = np.zeros((num_particles, 4), dtype=np.float32)
pos[:, 3] = 1.0
pos[:, 1] = 0.95*radius
pos[:, 2] = 0.95*radius
for i in range(num_particles):
    pos[i, 0] = L/2 + (i-num_particles/2) * 1.95*radius
sim.set_positions(pos)
vel = np.zeros((num_particles, 4), dtype=np.float32)
vel[:, 3] = 0.0
sim.set_velocities(vel) 
dt = 0.01

current_ov = sim.get_max_overlap()
print(f"initial overlap: {current_ov:<8.4f} pos: {pos[:,0]} ")

scales = np.empty(num_particles, dtype=np.float32)
scales[:] = 0.95*radius
scales[0] = 1.1*radius
sim.set_scales(scales)
for steps in range(10):
    sim.step(dt)
    current_ov = sim.get_max_overlap()
    steps += 1
    pos = sim.get_positions()   
    vel = sim.get_velocities()
    print(f"{steps:<5} overlap: {current_ov:<8.4f} pos: {pos[:,0]} ")
    #print(f"{steps:<5} overlap: {current_ov:<8.4f}")
